# Cleaning2
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

client = DatalakeClient()

# Get the files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
search = client.query_files(
    query={'custom.level' : 'cleaned_02', 'custom.source' : 'ADNI'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

## Operazioni
- Trasformare i volumi come percentuali di ICV

In [3]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned2'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned3'

In [4]:
create_new_support_file(support_file, support_file_path, new_name=new_name, rename=False)

In [5]:
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')
new_support_file = pd.read_excel(new_name+'.xlsx')

In [6]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    # Transform volumes as ICV percentage
    final_df, file_code, metadata_costum = dataCleaner.transform_volumes_as_ICV_percent(df_new, file_name, prefix='cleaned/single_file')
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)

    # Create new file name for datalake
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_03')
    # Update file level
    new_metadata = metadata_costum['level'] = 'cleaned_03'

    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=metadata_costum
    )
    
save_df(df_to_save=new_support_file, output_path=new_name)     



 ---- ADNIMERGE_25Jul2025_02.csv


 ---- MMSE_25Jul2025_02.csv
No volumes columns found for file code MMSE


 ---- PTDEMOG_25Jul2025_02.csv
No volumes columns found for file code PTDEMOG


 ---- ADSP_PHC_BIOMARKER_25Jul2025_02.csv
No volumes columns found for file code ADSP_PHC_BIOMARKER


 ---- BLCHANGE_25Jul2025_02.csv
No volumes columns found for file code BLCHANGE


 ---- DXSUM_25Jul2025_02.csv
No volumes columns found for file code DXSUM
